In [1]:
from dataclasses import dataclass
from typing import Any, Dict, List

import numpy as np

from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import Normalizer

from transformers import pipeline as hf_pipeline


@dataclass
class Document:
    """
    Represents one source document.
    """

    text: str
    metadata: Dict[str, Any]


@dataclass
class Chunk:
    """
    Represents a smaller section of a document.
    """

    text: str
    metadata: Dict[str, Any]


class SVDDocumentRetriever:
    """
    Retrieves document chunks using:

    TF-IDF
        -> TruncatedSVD
        -> Normalizer
        -> Cosine similarity
    """

    def __init__(
        self,
        n_components: int = 100,
        chunk_size: int = 80,
        chunk_overlap: int = 20
    ):
        if chunk_overlap >= chunk_size:
            raise ValueError(
                "chunk_overlap must be smaller than chunk_size."
            )

        self.n_components = n_components
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

        self.chunks: List[Chunk] = []
        self.document_vectors = None
        self.retrieval_pipeline = None

    def chunk_document(self, document: Document) -> List"""
        Splits a document into overlapping word-based chunks.
        """

        words = document.text.split()
        chunks = []

        start = 0
        chunk_number = 1

        while start < len(words):
            end = start + self.chunk_size
            chunk_words = words[start:end]

            if not chunk_words:
                break

            chunk_text = " ".join(chunk_words)

            chunk_metadata = {
                **document.metadata,
                "chunk_number": chunk_number,
                "start_word": start,
                "end_word": min(end, len(words))
            }

            chunks.append(
                Chunk(
                    text=chunk_text,
                    metadata=chunk_metadata
                )
            )

            if end >= len(words):
                break

            start += self.chunk_size - self.chunk_overlap
            chunk_number += 1

        return chunks

    def create_chunks(self, documents: List[Document]) -> List"""
        Chunks all supplied documents.
        """

        self.chunks = []

        for document in documents:
            document_chunks = self.chunk_document(document)
            self.chunks.extend(document_chunks)

        return self.chunks

    def build_index(self, documents: List[Document]) -> None:
        """
        Creates chunks and fits the TF-IDF, SVD, and normalization
        retrieval pipeline.
        """

        self.create_chunks(documents)

        if len(self.chunks) < 2:
            raise ValueError(
                "At least two chunks are required to build the index."
            )

        chunk_texts = [chunk.text for chunk in self.chunks]

        # TruncatedSVD cannot have more useful components than the
        # number of training samples minus one.
        safe_components = min(
            self.n_components,
            len(chunk_texts) - 1
        )

        safe_components = max(1, safe_components)

        self.retrieval_pipeline = SklearnPipeline(
            steps=[
                (
                    "tfidf",
                    TfidfVectorizer(
                        lowercase=True,
                        stop_words="english",
                        ngram_range=(1, 2),
                        min_df=1,
                        sublinear_tf=True
                    )
                ),
                (
                    "svd",
                    TruncatedSVD(
                        n_components=safe_components,
                        random_state=42
                    )
                ),
                (
                    "normalizer",
                    Normalizer(
                        norm="l2",
                        copy=False
                    )
                )
            ]
        )

        self.document_vectors = (
            self.retrieval_pipeline.fit_transform(chunk_texts)
        )

        print("Index successfully created.")
        print(f"Number of chunks: {len(self.chunks)}")
        print(
            "Document-vector shape:",
            self.document_vectors.shape
        )

    def retrieve(
        self,
        question: str,
        top_k: int = 3
    ) -> List[Dict[str, Any]]:
        """
        Finds chunks most relevant to a question.
        """

        if self.retrieval_pipeline is None:
            raise RuntimeError(
                "The index has not been created. "
                "Call build_index() first."
            )

        if not question.strip():
            raise ValueError("The question cannot be empty.")

        # Apply the already-fitted TF-IDF -> SVD -> Normalizer
        # pipeline to the question.
        query_vector = self.retrieval_pipeline.transform([question])

        # The vectors have unit length because of Normalizer.
        # Therefore, their dot product is cosine similarity.
        similarity_scores = np.dot(
            self.document_vectors,
            query_vector.T
        ).ravel()

        top_k = min(top_k, len(self.chunks))

        top_indices = np.argsort(similarity_scores)[::-1][:top_k]

        results = []

        for rank, index in enumerate(top_indices, start=1):
            chunk = self.chunks[index]

            results.append(
                {
                    "rank": rank,
                    "score": float(similarity_scores[index]),
                    "text": chunk.text,
                    "metadata": chunk.metadata
                }
            )

        return results


class SVDRAG:
    """
    Complete Retrieval-Augmented Generation system.
    """

    def __init__(
        self,
        generator_model: str = "google/flan-t5-small",
        n_components: int = 100,
        chunk_size: int = 80,
        chunk_overlap: int = 20
    ):
        self.retriever = SVDDocumentRetriever(
            n_components=n_components,
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

        # This is the Transformers generation pipeline.
        # It is different from sklearn.pipeline.Pipeline.
        self.generator = hf_pipeline(
            task="text2text-generation",
            model=generator_model,
            tokenizer=generator_model
        )

    def index(self, documents: List[Document]) -> None:
        """
        Builds the retrieval index.
        """

        self.retriever.build_index(documents)

    @staticmethod
    def build_prompt(
        question: str,
        retrieved_results: List[Dict[str, Any]]
    ) -> str:
        """
        Combines retrieved context and the user's question.
        """

        context_sections = []

        for result in retrieved_results:
            source = result["metadata"].get(
                "source",
                "Unknown source"
            )

            section = (
                f"[Source: {source}]\n"
                f"{result['text']}"
            )

            context_sections.append(section)

        context = "\n\n".join(context_sections)

        prompt = f"""
Answer the question using only the provided context.

If the answer is not present in the context, respond exactly:
I do not have enough information in the provided documents.

Do not invent information.
Keep the answer clear and concise.

Context:
{context}

Question:
{question}

Answer:
""".strip()

        return prompt

    def ask(
        self,
        question: str,
        top_k: int = 3,
        max_new_tokens: int = 150
    ) -> Dict[str, Any]:
        """
        Retrieves context and generates a grounded answer.
        """

        retrieved_results = self.retriever.retrieve(
            question=question,
            top_k=top_k
        )

        prompt = self.build_prompt(
            question=question,
            retrieved_results=retrieved_results
        )

        generated_output = self.generator(
            prompt,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

        answer = generated_output[0]["generated_text"].strip()

        sources = []

        for result in retrieved_results:
            metadata = result["metadata"]

            source_information = {
                "source": metadata.get("source", "Unknown source"),
                "chunk_number": metadata.get("chunk_number"),
                "similarity_score": round(result["score"], 4),
                "retrieved_text": result["text"]
            }

            sources.append(source_information)

        return {
            "question": question,
            "answer": answer,
            "sources": sources,
            "prompt": prompt
        }


def main():
    documents = [
        Document(
            text="""
            Retrieval-Augmented Generation, commonly called RAG,
            combines document retrieval with language generation.
            When a user asks a question, the retriever searches an
            external knowledge collection for relevant passages.
            These passages are added to the language model's prompt.
            The model then generates an answer using the retrieved
            evidence. RAG can reduce unsupported answers and makes it
            possible to use private or recently updated information
            without retraining the complete language model.
            """,
            metadata={
                "source": "rag_introduction.txt",
                "topic": "RAG"
            }
        ),
        Document(
            text="""
            TF-IDF means term frequency-inverse document frequency.
            It represents a document using the importance of its words.
            Term frequency measures how often a term appears in one
            document. Inverse document frequency reduces the importance
            of terms that occur in many documents. TF-IDF produces
            sparse vectors and is commonly used for information
            retrieval, document classification, and keyword extraction.
            """,
            metadata={
                "source": "tfidf_notes.txt",
                "topic": "TF-IDF"
            }
        ),
        Document(
            text="""
            TruncatedSVD is a dimensionality-reduction algorithm.
            Unlike standard principal component analysis, TruncatedSVD
            can operate efficiently on sparse matrices such as TF-IDF
            matrices. Applying TruncatedSVD to term-document data is
            commonly associated with latent semantic analysis.
            It creates smaller dense vectors representing important
            latent patterns in the original documents.
            """,
            metadata={
                "source": "svd_notes.txt",
                "topic": "SVD"
            }
        ),
        Document(
            text="""
            A normalizer scales each sample independently so that its
            vector has unit norm. L2 normalization divides each vector
            by its Euclidean length. Once the document and query vectors
            are L2 normalized, their dot product corresponds to cosine
            similarity. Similar vectors receive larger similarity scores.
            """,
            metadata={
                "source": "normalization_notes.txt",
                "topic": "Normalization"
            }
        ),
        Document(
            text="""
            A machine learning pipeline chains several transformations.
            During training, the pipeline fits each required component
            in sequence. During inference, the same fitted transformations
            are applied to new input. In a text-retrieval system, a
            pipeline can connect TF-IDF vectorization, dimensionality
            reduction, and vector normalization.
            """,
            metadata={
                "source": "pipeline_notes.txt",
                "topic": "Pipeline"
            }
        )
    ]

    rag = SVDRAG(
        generator_model="google/flan-t5-small",
        n_components=50,
        chunk_size=50,
        chunk_overlap=10
    )

    rag.index(documents)

    print("\nType 'exit' to stop the program.")

    while True:
        question = input("\nEnter your question: ").strip()

        if question.lower() in {"exit", "quit"}:
            print("Program stopped.")
            break

        if not question:
            print("Please enter a question.")
            continue

        result = rag.ask(
            question=question,
            top_k=3,
            max_new_tokens=150
        )

        print("\nAnswer:")
        print(result["answer"])

        print("\nRetrieved sources:")

        for source in result["sources"]:
            print(
                f"- {source['source']} | "
                f"chunk {source['chunk_number']} | "
                f"score {source['similarity_score']}"
            )


if __name__ == "__main__":
    main()

SyntaxError: expected ':' (3829965282.py, line 63)